# HW03 - 卷积和池化层

丁平尖
2026年6月2日

## 2 卷积和池化层

### 2.1 理论计算题

输入一张大小为 $3\times32\times32$（通道数 $\times$ 高 $\times$ 宽）的彩色图像。通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 $3\times5\times5$。设定填充（Padding）为 2，步幅（Stride）为 2。

**1. 计算该卷积层输出的特征图（Feature Map）的尺寸**

**解答：**

输出特征图尺寸的计算公式：

$$H_{out} = \lfloor \frac{H_{in} - K + 2P}{S} \rfloor + 1$$

其中：
- $H_{in} = 32$（输入高/宽）
- $K = 5$（卷积核尺寸）
- $P = 2$（填充）
- $S = 2$（步幅）

$$H_{out} = \lfloor \frac{32 - 5 + 2 \times 2}{2} \rfloor + 1 = \lfloor \frac{31}{2} \rfloor + 1 = 15 + 1 = 16$$

输出通道数 = 卷积核个数 = **16**

**输出特征图尺寸：$16 \times 16 \times 16$**（通道数 $\times$ 高 $\times$ 宽）

**2. 计算单个输出通道的一个像素值所需的点乘（乘法）操作次数**

**解答：**

每个卷积核的大小为 $3 \times 5 \times 5$（输入通道数 $\times$ 高 $\times$ 宽）。

单个输出像素值需要将整个卷积核与对应输入区域做逐元素相乘再求和，因此：

$$\text{乘法次数} = C_{in} \times K_h \times K_w = 3 \times 5 \times 5 = 75 \text{ 次}$$

### 2.2 编程题 - 手动实现二维最大池化

In [ ]:
import numpy as np

def max_pool2d_forward(input_array, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化（Max Pooling）前向传播函数。
    不使用深度学习框架的底层 Pooling API，仅使用 NumPy。

    参数:
        input_array: numpy array, shape (N, C, H, W) 或 (C, H, W) 或 (H, W)
        kernel_size: int, 池化核大小
        stride: int, 步幅，默认为 1
        padding: int, 填充大小，默认为 0

    返回:
        output: numpy array, 池化后的输出
    """
    # 确保输入是 4D: (N, C, H, W)
    if input_array.ndim == 2:
        input_array = input_array[np.newaxis, np.newaxis, :, :]
        squeeze_output = True
    elif input_array.ndim == 3:
        input_array = input_array[np.newaxis, :, :, :]
        squeeze_output = True
    else:
        squeeze_output = False

    N, C, H, W = input_array.shape

    # 添加填充
    if padding > 0:
        input_padded = np.pad(input_array,
                              ((0, 0), (0, 0), (padding, padding), (padding, padding)),
                              mode='constant', constant_values=0)
        H_padded, W_padded = H + 2 * padding, W + 2 * padding
    else:
        input_padded = input_array
        H_padded, W_padded = H, W

    # 计算输出尺寸
    H_out = (H_padded - kernel_size) // stride + 1
    W_out = (W_padded - kernel_size) // stride + 1

    output = np.zeros((N, C, H_out, W_out))

    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            h_end = h_start + kernel_size
            w_start = j * stride
            w_end = w_start + kernel_size
            window = input_padded[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2, 3))

    if squeeze_output:
        output = output.squeeze()

    return output


# 测试示例
np.random.seed(42)
test_input = np.random.randint(0, 10, (1, 1, 6, 6)).astype(np.float64)
print("输入 (1, 1, 6, 6):")
print(test_input[0, 0])

print("\n--- Max Pooling 测试 ---")

# 测试 1: kernel_size=2, stride=2, padding=0
output1 = max_pool2d_forward(test_input, kernel_size=2, stride=2, padding=0)
print("\nkernel_size=2, stride=2, padding=0")
print("输出 shape:", output1.shape)
print("输出:")
print(output1[0, 0])

# 测试 2: kernel_size=3, stride=1, padding=1
output2 = max_pool2d_forward(test_input, kernel_size=3, stride=1, padding=1)
print("\nkernel_size=3, stride=1, padding=1")
print("输出 shape:", output2.shape)
print("输出:")
print(output2[0, 0])

# 验证: 使用 PyTorch 的 MaxPool2d 对比
import torch
import torch.nn as nn

x_torch = torch.tensor(test_input, dtype=torch.float32)

pool1 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
out1_ref = pool1(x_torch).numpy()
print("\nPyTorch 对比 (kernel=2, stride=2, pad=0) - 一致:", np.allclose(output1, out1_ref))

pool2 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
out2_ref = pool2(x_torch).numpy()
print("PyTorch 对比 (kernel=3, stride=1, pad=1) - 一致:", np.allclose(output2, out2_ref))

## 3 LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

在 VGG 网络中，作者频繁使用多个 $3\times3$ 卷积核级联来代替较大的卷积核（如 $5\times5$ 或 $7\times7$）。假设输入和输出的特征图通道数均为 $C$。

**1. 计算一个 $5\times5$ 卷积层（不带偏置）的参数量：**

$$\text{参数量} = C \times C \times 5 \times 5 = 25C^2$$

**2. 计算两个串联的 $3\times3$ 卷积层（不带偏置，两层通道数都为 $C$）的总参数量：**

$$\text{参数量} = C \times C \times 3 \times 3 + C \times C \times 3 \times 3 = 9C^2 + 9C^2 = 18C^2$$

**结论：** 两个 $3\times3$ 卷积层的参数量 $18C^2$ 小于一个 $5\times5$ 卷积层的参数量 $25C^2$，同时感受野相同（都是 $5\times5$），但引入了更多的非线性变换，因此 VGG 采用这种设计更优。

### 3.2 编程题 - NiN Block

In [ ]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块：一个普通卷积层 + 两个 1x1 卷积层，每层后接 ReLU 激活。

    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数（也是最后一个 1x1 卷积的输出通道数）
        kernel_size: 普通卷积层的卷积核大小
        stride: 普通卷积层的步幅
        padding: 普通卷积层的填充
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.net = nn.Sequential(
            # 第一层：普通卷积 + ReLU
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(inplace=True),
            # 第二层：1x1 卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, 1, 1, 0),
            nn.ReLU(inplace=True),
            # 第三层：1x1 卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, 1, 1, 0),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


# 测试 NiN Block
nin_block = NiNBlock(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=2)
print("NiN Block 结构:")
print(nin_block.net)

x = torch.randn(1, 3, 224, 224)
y = nin_block(x)
print("\n输入形状:", x.shape)
print("输出形状:", y.shape)

## 4 Inception, 批量归一化和残差网络

### 4.1 理论计算题

在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：$x_1=2, x_2=4, x_3=6, x_4=8$。假设当前批量归一化层学到的缩放参数 $\gamma=2$，平移参数 $\beta=1$，常数 $\epsilon=0$。

**Batch Normalization 计算过程：**

**第一步：计算均值**
$$\mu = \frac{x_1 + x_2 + x_3 + x_4}{4} = \frac{2 + 4 + 6 + 8}{4} = 5$$

**第二步：计算方差**
$$\sigma^2 = \frac{1}{4} \left[(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2\right] = \frac{9+1+1+9}{4} = 5$$

**第三步：归一化**
$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{x_i - 5}{\sqrt{5}}$$

**第四步：缩放和偏移** $y_i = \gamma \hat{x}_i + \beta$:

$$y_1 = 2 \times \frac{2-5}{\sqrt{5}} + 1 = 1 - \frac{6}{\sqrt{5}} \approx -1.6833$$

$$y_2 = 2 \times \frac{4-5}{\sqrt{5}} + 1 = 1 - \frac{2}{\sqrt{5}} \approx 0.1056$$

$$y_3 = 2 \times \frac{6-5}{\sqrt{5}} + 1 = 1 + \frac{2}{\sqrt{5}} \approx 1.8944$$

$$y_4 = 2 \times \frac{8-5}{\sqrt{5}} + 1 = 1 + \frac{6}{\sqrt{5}} \approx 3.6833$$

In [ ]:
import numpy as np

x = np.array([2.0, 4.0, 6.0, 8.0])
gamma = 2.0
beta = 1.0
eps = 0.0

mu = np.mean(x)
var = np.mean((x - mu)**2)

print(f"均值 mu = {mu}")
print(f"方差 sigma^2 = {var}")

x_hat = (x - mu) / np.sqrt(var + eps)
y = gamma * x_hat + beta

print("\n归一化结果:")
for i in range(len(x)):
    print(f"  y_{i+1} = {y[i]:.6f}")

print("\n精确值:")
print(f"  y_1 = 1 - 6/sqrt(5) = {1 - 6/np.sqrt(5):.6f}")
print(f"  y_2 = 1 - 2/sqrt(5) = {1 - 2/np.sqrt(5):.6f}")
print(f"  y_3 = 1 + 2/sqrt(5) = {1 + 2/np.sqrt(5):.6f}")
print(f"  y_4 = 1 + 6/sqrt(5) = {1 + 6/np.sqrt(5):.6f}")

### 4.2 编程题 - 残差块（Residual Block）

In [ ]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块 (Residual Block)

    参数:
        input_channels: 输入通道数
        output_channels: 输出通道数
        use_1x1conv: 是否使用 1x1 卷积来调整输入的通道数和形状
        stride: 第一个卷积层的步幅
    """
    def __init__(self, input_channels, output_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, output_channels, kernel_size=3,
                               padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(output_channels)
        self.conv2 = nn.Conv2d(output_channels, output_channels, kernel_size=3,
                               padding=1, stride=1)
        self.bn2 = nn.BatchNorm2d(output_channels)

        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, output_channels,
                                   kernel_size=1, stride=stride)
        else:
            self.conv3 = None

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.conv3 is not None:
            x = self.conv3(x)

        out += x  # 残差连接
        return torch.relu(out)


# 测试 Residual Block
print("=== 测试 1: input_channels == output_channels, 无 1x1 conv ===")
blk1 = Residual(3, 3)
x1 = torch.randn(2, 3, 16, 16)
print("输入形状:", x1.shape)
print("输出形状:", blk1(x1).shape)
print(blk1)

print("\n=== 测试 2: input_channels != output_channels, 使用 1x1 conv ===")
blk2 = Residual(3, 6, use_1x1conv=True, stride=2)
print("输入形状:", x1.shape)
print("输出形状:", blk2(x1).shape)
print(blk2)

## 5 图像增广，微调和样式迁移

### 5.1 理论计算题

**1. 为什么底层特征提取层使用较小学习率，而顶层输出层使用较大学习率？**

- **底层特征提取层**：在预训练时已经学到了通用的视觉特征（如边缘、纹理、形状等），这些特征在源数据集和目标数据集之间通常是可迁移的。如果使用较大学习率，会破坏这些已经学好的有用特征（灾难性遗忘）。使用较小的学习率（或冻结参数），可以在保留已有知识的同时进行微调。
- **顶层输出层**：是与具体任务相关的层，在预训练时是随机初始化的或针对源任务训练的，需要重新学习以适应新的目标类别。因此需要较大学习率来快速收敛到新的任务。

**2. 如果目标数据集非常小且与源数据集非常相似的微调策略：**

- **冻结大部分底层参数**，只训练最后几层或仅训练顶层分类器，减少可训练参数数量以防止过拟合。
- **使用较小的学习率**，避免在小数据集上过度更新权重。
- **使用数据增广**（Data Augmentation）来人为增大小数据集的多样性。
- **使用正则化技术**，如 Dropout、权重衰减（Weight Decay）等。
- **采用 Early Stopping**，在验证集性能不再提升时及时停止训练。
- **只微调少量层**，如仅微调最后 1-2 个卷积层的参数。

### 5.2 编程题 - 图像增广管道

In [ ]:
import torch
import torchvision.transforms as transforms

# 创建图像增广管道
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪，面积比例在 0.08 到 1.0 之间，然后缩放到 224x224
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度、对比度和饱和度，范围为 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4. 转换为 PyTorch 张量
    transforms.ToTensor(),
])

print("图像增广管道:")
print(augmentation_pipeline)

# 测试
from PIL import Image
import numpy as np

# 创建一个随机测试图像
test_img = Image.fromarray(np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8))
augmented = augmentation_pipeline(test_img)
print("\n原始图像大小:", test_img.size, "(W x H)")
print("增广后张量形状:", augmented.shape, "(C x H x W)")
print("像素值范围: [{:.4f}, {:.4f}]".format(augmented.min(), augmented.max()))

## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

计算 IoU：真实框 A=[10,10,50,50]，预测框 B=[30,30,70,70]

**交并比（IoU）计算：**

**第一步：计算交集区域**
- 交集左上角：$(\max(10,30), \max(10,30)) = (30, 30)$
- 交集右下角：$(\min(50,70), \min(50,70)) = (50, 50)$
- 交集面积 = $(50-30) \times (50-30) = 20 \times 20 = 400$

**第二步：计算并集区域**
- A 的面积 = $(50-10) \times (50-10) = 40 \times 40 = 1600$
- B 的面积 = $(70-30) \times (70-30) = 40 \times 40 = 1600$
- 并集面积 = $1600 + 1600 - 400 = 2800$

**第三步：计算 IoU**
$$IoU = \frac{\text{交集面积}}{\text{并集面积}} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429$$

In [ ]:
import numpy as np

def compute_iou(box_a, box_b):
    """
    计算两个边界框的 IoU。
    格式: [左上角x, 左上角y, 右下角x, 右下角y]
    """
    x1_inter = max(box_a[0], box_b[0])
    y1_inter = max(box_a[1], box_b[1])
    x2_inter = min(box_a[2], box_b[2])
    y2_inter = min(box_a[3], box_b[3])

    inter_width = max(0, x2_inter - x1_inter)
    inter_height = max(0, y2_inter - y1_inter)
    inter_area = inter_width * inter_height

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])

    union_area = area_a + area_b - inter_area
    iou = inter_area / union_area if union_area > 0 else 0
    return iou, inter_area, union_area

A = [10, 10, 50, 50]
B = [30, 30, 70, 70]

iou, inter_area, union_area = compute_iou(A, B)

print("真实框 A =", A)
print("预测框 B =", B)
print("\n交集面积 =", inter_area)
print("A 面积 =", (A[2]-A[0])*(A[3]-A[1]))
print("B 面积 =", (B[2]-B[0])*(B[3]-B[1]))
print("并集面积 =", union_area)
print("\nIoU = {}/{} = {:.6f}".format(inter_area, union_area, iou))
print("IoU = 1/7 = {:.6f}".format(1/7))

### 6.2 编程题 - 标签平滑交叉熵损失

In [ ]:
import torch
import torch.nn as nn
import numpy as np

def label_smoothing_cross_entropy(pred, target, num_classes, epsilon=0.1):
    """
    计算标签平滑后的交叉熵损失。

    参数:
        pred: 模型输出 logits, shape (N, K), N 为样本数, K 为类别数
        target: 真实标签, shape (N,), 值为类别索引
        num_classes: 总类别数 K
        epsilon: 平滑因子，默认 0.1

    返回:
        loss: 标签平滑交叉熵损失值（标量）
    """
    # 构建标签平滑后的目标分布
    smooth_target = torch.full_like(pred, epsilon / (num_classes - 1))
    smooth_target.scatter_(1, target.unsqueeze(1), 1.0 - epsilon)

    # 计算 log softmax
    log_probs = torch.log_softmax(pred, dim=1)

    # 计算交叉熵: -sum(y_smooth * log(y_pred))
    loss = -torch.sum(smooth_target * log_probs, dim=1)

    # 取平均
    return loss.mean()


# 测试
torch.manual_seed(42)
num_classes = 5
N = 3

pred = torch.randn(N, num_classes)
target = torch.tensor([0, 2, 4])

print("模型输出 logits:")
print(pred)
print("\n真实标签:", target.tolist())
print("类别数:", num_classes, ", 平滑因子: epsilon = 0.1")

# 自实现的标签平滑交叉熵
loss_custom = label_smoothing_cross_entropy(pred, target, num_classes, epsilon=0.1)
print("\n自实现标签平滑交叉熵损失: {:.6f}".format(loss_custom.item()))

# 使用 PyTorch 内置函数对比
loss_ref = nn.CrossEntropyLoss(label_smoothing=0.1)(pred, target)
print("PyTorch 内置交叉熵损失:   {:.6f}".format(loss_ref.item()))
print("两者一致:", torch.allclose(loss_custom, loss_ref, atol=1e-6))

# 展示标签平滑的效果
K = 5
epsilon = 0.1
target_example = torch.tensor([2])
smooth = torch.full((1, K), epsilon / (K - 1))
smooth.scatter_(1, target_example.unsqueeze(1), 1.0 - epsilon)
print("\n标签平滑示例 (K={}, epsilon={}):".format(K, epsilon))
print("  真实类别 =", target_example.item())
print("  平滑后目标分布 =", smooth[0].tolist())